# A2: Stability

Do the decision models give the same answer when the same request is presented differently? Every model receives the same 500 sampled requests per dataset (per-column and joint). One factor changes at a time: unchanged replay (three executions), three reviewed rewordings of the question sentence, reversed text-field order (joint requests with at least two non-empty fields) and reversed option order (multiclass, realigned by class identity). Changes are reported relative to replay noise, within datasets first and then with equal dataset weight. There are no lexical rewordings.

All tables come from **one compact export**, `output_JEVPFN/exp02_robustness/results/robustness/robustness.zip`, built on VSC by the Mindwell CPU job `scripts/slurm/rq1_robustness_export.slurm` (`python -m src.evaluation.robustness_export`) from the completed campaigns on PROJECT.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import matplotlib.pyplot as plt
from src.utils.notebook_display import display_local, display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import study_export, study_reports
from src.visualize import figures, style
from src.visualize.stability import wording_averaging_figure
style.apply()
NOTEBOOK = "exp02_robustness"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. Scope, provenance and the export read by this notebook")
display_json("Notebook execution identity", provenance())
frames, receipt = study_reports.robustness_export_tables()
print(study_export.describe(receipt))
record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, "Export identity")
if frames:
    display_table("Completed campaigns in the export", frames["sources"], preview_rows=8)

Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\exp02_robustness\results\robustness\robustness.zip is absent. No measurements are shown.


## Historical 0A sensitivity diagnostics

These historical probes use three real inputs (one per task), including option-order and embedded-instruction interventions. They are not the 500-request stability study; an embedded instruction also changes the supplied text. The figure compares the full probability vectors; an asterisk marks a changed top answer. This is a 0A diagnostic to report, never a criterion for choosing prompts from test results. The probes travel in the same export ZIP; no receipt is read locally.


In [2]:
report_section('2. Historical 0A sensitivity diagnostics')
from src.visualize.research import robustness_matrix_figure
if not frames:
    print("Waiting for export: no historical 0A sensitivity tables are available.")
else:
    changes = frames["historical_0a"]
    display_table("Every robustness comparison", changes)
    display_table("Dated robustness receipts (file name and SHA-256)", frames["historical_0a_sources"])
    for fig, name, caption in robustness_matrix_figure(changes):
        save(fig, name, caption=caption)
        display_local(fig)
        plt.close(fig)

Waiting for export: no historical 0A sensitivity tables are available.


## One factor at a time, relative to replay noise

For each model, request mode and factor: the mean total variation between the fresh original and the changed presentation, the share of changed top answers (exact ties excluded) and the excess over the unchanged replays on the same requests. Within-dataset values are combined with equal dataset weight.

In [3]:
report_section("3. One factor at a time, relative to replay noise")
if not frames:
    print("Waiting for export: no stability tables are available.")
else:
    summary = study_reports.equal_dataset_mode_effects(frames["mode_effects"])
    display_table("Change and excess over replay per model, mode and factor (equal dataset weight)", summary.round(4), preview_rows=12)
    display_table("Excess change above unchanged replays, all requests (equal dataset weight)", study_reports.equal_dataset_replay(frames["effects"]).round(4))
    display_table("Coverage, including field orders normalized by the model", frames["coverage"], preview_rows=8)
    record(summary, "Mode-specific changes relative to replay, equal dataset weight")
    record(frames["mode_effects"], "Mode-specific changes per dataset")

Waiting for export: no stability tables are available.


## Averaging over presentations

Does averaging over presentations help? This uses only responses already collected above; no model is called again. (a) Wording: the distance between the averages of two disjoint pairs of wordings, {original, rewording 1} and {rewording 2, rewording 3}, compared with the distance between single wordings across those pairs. A ratio below one means averaging reduces wording sensitivity. (b) Stand-alone quality on the sampled *joint* requests (labels used only for scoring): the original wording against the average of the four wordings, and the original option or field order against the average of original and reversed order.

In [4]:
report_section("4. Averaging over presentations")
if not frames:
    print("Waiting for export: no averaging tables are available.")
else:
    wording = study_reports.wording_averaging_summary(frames["wording_averaging"])
    for fig, name, caption in wording_averaging_figure(wording):
        save(fig, name, caption=caption)
        display_local(fig)
        plt.close(fig)
    display_table("Wording distance: single versions versus averaged pairs (equal dataset weight)", wording.round(4), preview_rows=12)
    quality = study_reports.averaging_quality_summary(frames["averaging_quality"])
    display_table("Stand-alone quality: averaged versus standard presentation (primary; log loss, regression RPS) and versus mean single presentation (Jensen gap, guaranteed for convex scores); sampled joint requests, equal dataset weight", quality.round(4), preview_rows=12)
    record(wording, "Wording averaging, equal dataset weight")
    record(quality, "Averaged-presentation quality, equal dataset weight")
    record(frames["averaging_quality"], "Averaged-presentation quality per dataset")

Waiting for export: no averaging tables are available.


## Complete report

In [5]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified export was read; no inference, model training or live VSC changes were performed.")

exp02_robustness

1. Scope, provenance and the export read by this notebook


Notebook execution identity
{
  "code_sha256": "89bab8f7f37ddbae29ac090298848affb33a3d2a7907e47b68f23044a3c50c4e",
  "git_head": "96e484c27b0403005cbc5d4b92f2ca0b45e31bd9",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Export identity
{
  "status": "awaiting_export",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\exp02_robustness\\results\\robustness\\robustness.zip",
  "created_utc": null,
  "origin": null,
  "sha256": null,
  "bytes": null,
  "schema": null,
  "receipt_s